<a href="https://colab.research.google.com/github/almagomen/v2v/blob/monorepo-subtree/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video – Despliegue en Colab

1. **Activa T4 GPU**.
2. Ejecuta celdas.
3. Configura **Secrets**.

In [ ]:
# @title ⚠️ Ejecutar APP (V2V - Sincronización Total)
import os, time, threading, sys, warnings, subprocess
import uvicorn
from pathlib import Path
from google.colab import userdata, drive, output
warnings.filterwarnings("ignore", category=UserWarning); warnings.filterwarnings("ignore", category=SyntaxWarning)

# --- 1. CONFIGURACIÓN ---
if not os.path.exists("/content/drive"): drive.mount("/content/drive")

GHP_TOKEN = userdata.get("GHP_GITHUB_TOKEN")
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git"
REPO_PATH = Path("/content/v2v")
BRANCH_NAME = "monorepo-subtree"
BACKEND_PORT, FRONTEND_PORT = 8000, 5173
LOG_FILE = REPO_PATH / "backend.log"

# --- 2. ACTUALIZACIÓN INTELIGENTE (Solo si hay cambios) ---
if not REPO_PATH.exists():
    print("📥 Clonando repositorio..."); 
    !git clone -q -b {BRANCH_NAME} {REPO_URL} {REPO_PATH}
else:
    !git -C {REPO_PATH} reset --hard HEAD -q
    !git -C {REPO_PATH} fetch origin {BRANCH_NAME} -q
    diff_output = !git -C {REPO_PATH} diff --name-status HEAD origin/{BRANCH_NAME}
    
    if diff_output:
        info_commit = !git -C {REPO_PATH} log origin/{BRANCH_NAME} -1 --pretty=format:"%h - %s (%cr)"
        tags = {"M": "📝 Modificado", "A": "✨ Añadido", "D": "🗑️ Borrado", "R": "🏷️ Renombrado"}
        print(f"\n🚀 ¡Cambios detectados!\n📌 Commit: {info_commit[0]}.")
        for line in diff_output:
            s, f = line.split(None, 1)
            print(f"   • {tags.get(s[0], '❓ ' + s[0])}: {f}")
        !git -C {REPO_PATH} reset --hard origin/{BRANCH_NAME} -q
    else:
        !git -C {REPO_PATH} reset --hard origin/{BRANCH_NAME} -q
        print("✔️ Repositorio al día.")

if str(REPO_PATH) not in sys.path: sys.path.insert(0, str(REPO_PATH))

# --- 3. CARGA PERSISTENTE DE MODELOS (Fuera del ciclo de la API) ---
if 'models_loaded' not in globals():
    print("📦 Preparando entorno e instalando dependencias...")
    !npm install -g pnpm ---silent
    !uv pip install git+https://github.com/jianfch/stable-ts.git git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install "yt-dlp[default]" bs-roformer-infer tinydb sentencex torchcodec==0.7.0 'torch-audiomentations>=0.12.0' 'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q
    !uv pip install openai-whisper 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' -q
    
    print("🛠️ Cargando modelos en GPU...")
    from src.stt import get_whisper_model; from src.diarization import get_diarization_model; from src.download import get_roformer_model
    # Guardamos instancias reales en globals
    globals().update({
        'shared_whisper': get_whisper_model(),
        'shared_diarization': get_diarization_model(),
        'shared_roformer': get_roformer_model(),
        'models_loaded': True
    })
    print("✅ Modelos listos en RAM.")
else:
    print("⚡ Modelos detectados en memoria. Saltando carga.")

# --- 4. CONFIGURACIÓN DE LOGS ---
log_config = {
    "version": 1, "disable_existing_loggers": False,
    "formatters": {
        "default": {"()": "uvicorn.logging.DefaultFormatter", "fmt": "%(levelprefix)s %(message)s", "use_colors": True},
        "access": {"()": "uvicorn.logging.AccessFormatter", "fmt": '%(levelprefix)s %(client_addr)s - "%(request_line)s" %(status_code)s', "use_colors": True},
    },
    "handlers": {
        "console": {"class": "logging.StreamHandler", "formatter": "default", "stream": "ext://sys.stdout"},
        "file": {"class": "logging.FileHandler", "formatter": "default", "filename": str(LOG_FILE), "mode": "w"},
        "access_console": {"class": "logging.StreamHandler", "formatter": "access", "stream": "ext://sys.stdout"},
    },
    "loggers": {
        "uvicorn": {"handlers": ["console", "file"], "level": "INFO", "propagate": False},
        "uvicorn.error": {"handlers": ["file"], "level": "INFO"},
        "uvicorn.access": {"handlers": ["access_console", "file"], "level": "INFO", "propagate": False},
    },
}

# --- 5. SERVIDOR BACKEND (Con inyección de modelos) ---
class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self): return

def run_backend():
    global server
    # Limpieza de módulos (excepto los pesados si fuera necesario)
    for m in [m for m in sys.modules if m.startswith(('api', 'src'))]:
        if 'models' in m: continue # Protegemos referencias a modelos
        del sys.modules[m]
    
    from api.main import app
    
    # Inyectar los modelos cargados en el estado de la APP
    # Esto evita que los servicios llamen a load_model() internamente
    app.state.whisper = globals().get('shared_whisper')
    app.state.diarization = globals().get('shared_diarization')
    app.state.roformer = globals().get('shared_roformer')

    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT, log_config=log_config, loop="asyncio", timeout_keep_alive=5)
    server = ControlledServer(config=config); server.run()

if 'server' in globals() and server.started:
    print("🛑 Reiniciando..."); server.should_exit = True; time.sleep(2)

threading.Thread(target=run_backend, daemon=True).start()

# --- 6. FRONTEND (Vite) ---
OPENREEL_PATH = REPO_PATH / "openreel"
print("🏗️ Preparando OpenReel..."); 
!cd {OPENREEL_PATH} && pnpm install -silent
!fuser -k {FRONTEND_PORT}/tcp 2>/dev/null
f_env = os.environ.copy(); f_env["FORCE_COLOR"] = "1"
process = subprocess.Popen(["pnpm", "dev", "--", "--host", "0.0.0.0", "--port", str(FRONTEND_PORT)],
    cwd=str(OPENREEL_PATH), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=f_env)

def log_reader(pipe):
    C, RS = "\033[1;36m", "\033[0m"
    with open(LOG_FILE, "a") as f:
        for line in iter(pipe.readline, ''):
            if line.strip():
                f.write(f"\n{C}[FRONTEND]{RS} {line.strip()}\n"); f.flush()

threading.Thread(target=log_reader, args=(process.stdout,), daemon=True).start()

# --- 7. LANZAMIENTO ---
print("\n✨ ENTORNO ACTUALIZADO Y LISTO ✨")
output.serve_kernel_port_as_window(FRONTEND_PORT, path="/", skip_warning=True)